# SDG 1: No poverty

The SDG 1 (No poverty) index of the SDG Atlas of Bolivia 2026 and its 4 indicators, multidimensional poverty, unsatisfied basic needs, energy poverty and access to the three basic services, for every municipality near 2012 and near 2024.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg01/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg01/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Map the change of multidimensional poverty (sdg1_2_mpi) between ~2012 and ~2024 and tell me which departments improved most."
- "Which 10 municipalities had the highest multidimensional poverty (sdg1_2_mpi) in 2024, and how did each change since ~2012?"
- "Plot unsatisfied basic needs (sdg1_4_nbi) against access to the three basic services (sdg1_4_ssb) in 2024 and describe the relationship."
- "Is poverty spatially clustered? Run geo.moran and geo.lisa on sdg1_2_mpi in 2024 and explain the clusters on a map."
- "Energy poverty (sdg1_4_eep) is measured in 2016 and 2024, not 2012. Check its reference years with geo.reference_years("sdg01") and compare its distribution in both periods by department."


## Variables

The columns of `sdg01`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("sdg01")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Period |  |
| `mun_id` | INE municipality code |  |
| `index_sdg1` | No poverty index (0-100) | index (0-100) |
| `sdg1_2_mpi` | Multidimensional poverty index (0-1) | index (0-1) |
| `sdg1_4_eep` | Energy poverty (% of dwellings) | % of dwellings |
| `sdg1_4_nbi` | Unsatisfied basic needs (% of population) | % of population |
| `sdg1_4_ssb` | Access to the 3 basic services (% of households) | % of households |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
# The SDG 1 index (0-100, higher is better), ~2024
gdf = geo.load("sdg01", variables="index_sdg1", years=2024, geometry=True)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg1", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg1": "No poverty index (0-100)"},
          title="SDG 1: No poverty index, ~2024").show()

# The two periods: initial (~2012) and final (~2024) values
df = geo.load("sdg01", variables="index_sdg1")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg1")
wide = wide.add_prefix("y")
wide["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(wide, x="y2012", y="y2024", hover_name="name",
           labels={"y2012": "Index, ~2012", "y2024": "Index, ~2024"},
           title="SDG 1: No poverty index, ~2012 and ~2024")

## Load the data without the package (plain URLs)

In [ ]:
# No geo4bolivia needed: every file is a plain URL
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
df = pd.read_csv(BASE + "data/sdg01/map343/sdg01.csv")  # geo_id, year, mun_id, ...
municipalities = gpd.read_file(BASE + "geo/boundaries/map343.geojson")

index = df.loc[df["year"] == 2024, ["geo_id", "index_sdg1"]]
gdf = municipalities.merge(index, on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg1", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg1": "No poverty index (0-100)"},
          title="SDG 1: No poverty index, ~2024")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/sdg01/map343/sdg01_wide.gpkg",
                    layer="sdg01")   # or sdg01_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg1_2024", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg1_2024": "No poverty index (0-100)"},
          title="No poverty index (0-100), ~2024")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/sdg01/map343/sdg01_wide.geojson",
                 column="index_sdg1_2024", name="No poverty index (0-100), ~2024",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## Map the change, ~2012 to ~2024

In [ ]:
df = geo.load("sdg01", variables="index_sdg1")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg1")
wide["change"] = wide[2024] - wide[2012]
gdf = geo.boundaries("map343").merge(wide["change"].reset_index(),
                                           on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="change", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"change": "Change in the index (points)"},
          title="SDG 1: change in the index, ~2012 to ~2024")

## Recompute the index from its indicators

In [ ]:
df = geo.load("sdg01", years=2024).set_index("geo_id")
bounds = {"sdg1_2_mpi": (0.563, 0), "sdg1_4_eep": (100, 0), "sdg1_4_nbi": (100, 0),
          "sdg1_4_ssb": (0, 100)}  # (worst, best)
scores = pd.DataFrame({v: 100 * (df[v] - w) / (b - w)
                       for v, (w, b) in bounds.items()}).clip(0, 100)
df["computed"] = scores.mean(axis=1)  # skips missing indicators
df["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(df, x="computed", y="index_sdg1", hover_name="name",
           labels={"computed": "Computed from the indicators",
                   "index_sdg1": "No poverty index (0-100)"},
           title="Recomputed vs published SDG 1 index, ~2024")

## Which indicators moved

In [ ]:
df = geo.load("sdg01")
bounds = {"sdg1_2_mpi": (0.563, 0), "sdg1_4_eep": (100, 0), "sdg1_4_nbi": (100, 0),
          "sdg1_4_ssb": (0, 100)}  # (worst, best)
labels = geo.dictionary("sdg01").set_index("variable")["label"]
rows = [{"indicator": labels[v], "period": f"~{year}",
         "score": (100 * (part[v] - w) / (b - w)).clip(0, 100).mean()}
        for year, part in df.groupby("year") for v, (w, b) in bounds.items()]
px.scatter(pd.DataFrame(rows), x="score", y="indicator", color="period",
           labels={"score": "Mean rescaled score (0-100)", "indicator": ""},
           title="SDG 1 indicators, ~2012 and ~2024")

## Rank the departments

In [ ]:
df = geo.load("sdg01", variables="index_sdg1")
dep = geo.aggregate(df, to="department",
                    rules={"index_sdg1": "weighted_mean:population"},
                    weights=geo.load("sdg", variables="population"))
dep["department"] = dep["geo_id"].map(
    geo.units("department").set_index("geo_id")["name"])
dep["period"] = "~" + dep["year"].astype(str)
px.scatter(dep.sort_values("index_sdg1"), x="index_sdg1", y="department",
           color="period",
           labels={"index_sdg1": "Index (population-weighted)"},
           title="SDG 1 index by department, ~2012 and ~2024")

## Poverty and night-time lights

In [ ]:
mpi = geo.load("sdg01", variables="sdg1_2_mpi", years=2012)
ntl = geo.load("ntl", variables="ln_t400NTLpc", years=2012)
df = mpi.merge(ntl, on=["geo_id", "year"])
df["name"] = df["geo_id"].map(geo.units("map343").set_index("geo_id")["name"])
px.scatter(df, x="ln_t400NTLpc", y="sdg1_2_mpi", hover_name="name",
           labels={"ln_t400NTLpc": "Log night-time lights per capita (trend)",
                   "sdg1_2_mpi": "Multidimensional poverty index"},
           title="Poverty and night-time lights, 2012")

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- Lykke E. Andersen and Fabiana Argandoña (SDSN Bolivia, Universidad Privada Boliviana): [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)
- SDSN Bolivia: [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La Paz: Universidad Privada Boliviana, SDSN Bolivia.
2. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
3. SDSN Bolivia (2025). Límites Municipales Bolivia 2025 [Archivo shapefile]. https://sdsnbolivia.org/datos-espaciales/
